# Convert PSD Results from Code Units to Physical (SI) Units

Loads the truncated PSD output from notebook 07 (hybrid-code normalized units)
and converts to SI units for a specified magnetic field strength B₀ and number
density n₀. Code normalization: B→B₀, length→dᵢ, time→Ωci⁻¹, velocity→V_A.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.constants import m_p, e, c, mu_0, epsilon_0
from pathlib import Path
import os

%matplotlib inline

In [ ]:
# ============================================================
# User-specified physical parameters — edit these
# ============================================================
B0 = 0.01       # T  (100 G, typical coronal loop)
n0 = 1e15       # m^-3  (10^9 cm^-3, typical coronal loop)

TRUNC_FILE = "lagrangian_batch_truncated.nc"
SAVE_NAME  = "lagrangian_batch_truncated_si.nc"
PSD_DIR    = os.path.join(os.path.dirname(os.getcwd()), "psd_plots_si") if os.path.basename(os.getcwd()) == "notebooks" else "psd_plots_si"
os.makedirs(PSD_DIR, exist_ok=True)

## Derive Physical Scales

In [ ]:
# Fundamental scales from B0 and n0
Omega_ci = e * B0 / m_p                          # ion cyclotron frequency [rad/s]
f_ci     = Omega_ci / (2 * np.pi)                # ion cyclotron frequency [Hz]
omega_pi = np.sqrt(n0 * e**2 / (epsilon_0 * m_p))  # ion plasma frequency [rad/s]
d_i      = c / omega_pi                          # ion inertial length [m]
V_A      = B0 / np.sqrt(mu_0 * n0 * m_p)        # Alfven speed [m/s]

print("Derived physical scales")
print("=" * 50)
print(f"  Omega_ci  = {Omega_ci:.4e} rad/s  ({f_ci:.4e} Hz)")
print(f"  omega_pi  = {omega_pi:.4e} rad/s")
print(f"  d_i       = {d_i:.4e} m  ({d_i*100:.4f} cm)")
print(f"  V_A       = {V_A:.4e} m/s  ({V_A/1e3:.2f} km/s)")
print(f"  V_A / c   = {V_A/c:.4e}")
print(f"  B0^2/mu_0 = {B0**2/mu_0:.4e} J/m^3")
print(f"  n0*m_p*V_A^2 = {n0*m_p*V_A**2:.4e} J/m^3  (should equal B0^2/mu_0)")

## Define Conversion Factors

| Quantity | Code unit | SI scale factor |
|----------|-----------|------------------|
| frequency | Ωci | f_Hz = f_code × Ωci |
| B field | B₀ | B_SI = B_code × B₀  [T] |
| E field | V_A B₀ | E_SI = E_code × V_A × B₀  [V/m] |
| Poynting (E×B) | n₀ mₚ V_A³ | S_SI = S_code × n₀ mₚ V_A³  [W/m²] |
| Energy density | n₀ mₚ V_A² = B₀²/μ₀ | u_SI = u_code × n₀ mₚ V_A²  [J/m³] |

For PSDs: PSD_SI = PSD_code × Q² / Ωci  (where Q is the field scale factor).

**Note on energy density:** The code stores uE = ½E² and uB = ½B² in code
units where the normalization absorbs ε₀ and 1/μ₀ factors (hybrid
approximation). The conversion n₀ mₚ V_A² applies to the code-unit energy
density as-is.

In [ ]:
# Field scale factors (code unit -> SI)
B_scale = B0                          # T
E_scale = V_A * B0                    # V/m
S_scale = n0 * m_p * V_A**3          # W/m^2  (Poynting: E x B, not E x B / mu_0)
u_scale = n0 * m_p * V_A**2          # J/m^3  (= B0^2 / mu_0)
freq_scale = Omega_ci                 # rad/s -> Hz conversion factor

# PSD conversion: PSD_SI = PSD_code * Q^2 / freq_scale
# Maps component name -> (psd_scale_factor, y-axis label)
CONV = {
    "Ex": (E_scale**2 / freq_scale, r"PSD [(V/m)$^2$/Hz]"),
    "Ey": (E_scale**2 / freq_scale, r"PSD [(V/m)$^2$/Hz]"),
    "Ez": (E_scale**2 / freq_scale, r"PSD [(V/m)$^2$/Hz]"),
    "Bx": (B_scale**2 / freq_scale, r"PSD [T$^2$/Hz]"),
    "By": (B_scale**2 / freq_scale, r"PSD [T$^2$/Hz]"),
    "Bz": (B_scale**2 / freq_scale, r"PSD [T$^2$/Hz]"),
    "Sx": (S_scale**2 / freq_scale, r"PSD [(W/m$^2$)$^2$/Hz]"),
    "Sy": (S_scale**2 / freq_scale, r"PSD [(W/m$^2$)$^2$/Hz]"),
    "Sz": (S_scale**2 / freq_scale, r"PSD [(W/m$^2$)$^2$/Hz]"),
    "uE": (u_scale**2 / freq_scale, r"PSD [(J/m$^3$)$^2$/Hz]"),
    "uB": (u_scale**2 / freq_scale, r"PSD [(J/m$^3$)$^2$/Hz]"),
}

print("PSD conversion factors (PSD_SI = factor * PSD_code):")
print("-" * 55)
for comp, (scale, label) in CONV.items():
    print(f"  {comp:4s}: {scale:.4e}   {label}")

## Load Data & Convert

In [ ]:
ds = xr.open_dataset(TRUNC_FILE)
print(ds)
print(f"\ndt (code) = {ds.attrs['dt']}")

# Convert frequency coordinate
freq_code = ds.frequency.values
freq_hz   = freq_code * freq_scale  # Hz

# Convert spatial coordinates
x0_si = ds.x0.values * d_i   # m
y0_si = ds.y0.values * d_i   # m

# Build SI dataset
ALL_COMPS = ["Ex", "Ey", "Ez", "Bx", "By", "Bz", "Sx", "Sy", "Sz", "uE", "uB"]

data_vars = {
    "end_time_idx": (["trajectory"], ds.end_time_idx.values),
    "x0_code":      (["trajectory"], ds.x0.values),
    "y0_code":      (["trajectory"], ds.y0.values),
    "x0":           (["trajectory"], x0_si),
    "y0":           (["trajectory"], y0_si),
    "n_freq":       (["trajectory"], ds.n_freq.values),
}

for comp in ALL_COMPS:
    scale, _ = CONV[comp]
    data_vars[f"psd_{comp}"] = (
        ["trajectory", "frequency"],
        ds[f"psd_{comp}"].values * scale,
    )

ds_si = xr.Dataset(
    data_vars,
    coords={
        "trajectory": ds.trajectory.values,
        "frequency":  freq_hz,
    },
    attrs={
        "source_file":  ds.attrs.get("source_file", ""),
        "psd_method":   ds.attrs.get("psd_method", ""),
        "dt_code":      ds.attrs["dt"],
        "dt_si":        ds.attrs["dt"] / Omega_ci,
        "B0_T":         B0,
        "n0_m3":        n0,
        "Omega_ci":     Omega_ci,
        "f_ci_Hz":      f_ci,
        "d_i_m":        d_i,
        "V_A_ms":       V_A,
        "freq_unit":    "Hz",
        "x0_unit":      "m",
        "y0_unit":      "m",
    },
)

ds.close()
print("\nConverted dataset:")
print(ds_si)

## Sanity Checks

In [ ]:
print("Sanity checks")
print("=" * 60)

# Frequency range
f_max = freq_hz[~np.isnan(freq_hz)].max()
print(f"Frequency range: 0 to {f_max:.2e} Hz  ({f_max/1e3:.2f} kHz)")
print(f"  f_ci = {f_ci:.2e} Hz  ({f_ci/1e3:.2f} kHz)")
print(f"  f_max / f_ci = {f_max / f_ci:.2f}")

# Peak PSD values for a representative trajectory
traj_idx = 0
n_f = int(ds_si.n_freq.values[traj_idx])
print(f"\nTrajectory {traj_idx} (n_freq={n_f}):")
for comp in ALL_COMPS:
    psd_vals = ds_si[f"psd_{comp}"].values[traj_idx, 1:n_f]  # skip DC
    valid = psd_vals[~np.isnan(psd_vals) & (psd_vals > 0)]
    if len(valid) > 0:
        ipeak = np.argmax(valid)
        f_peak = freq_hz[1:n_f][ipeak]
        _, label = CONV[comp]
        print(f"  {comp:4s}: peak = {valid[ipeak]:.4e}  at f = {f_peak:.4e} Hz  ({f_peak/1e3:.2f} kHz)")

# Parseval check: integral of PSD ~ variance
# (order-of-magnitude only — we only have the PSD, not the original signal)
print(f"\nParseval check (trajectory {traj_idx}, Bz):")
psd_bz = ds_si["psd_Bz"].values[traj_idx, :n_f]
f_arr  = freq_hz[:n_f]
df = f_arr[1] - f_arr[0]
integral = np.nansum(psd_bz) * df
print(f"  integral(PSD_Bz) * df = {integral:.4e} T^2")
print(f"  sqrt(integral)        = {np.sqrt(integral):.4e} T")
print(f"  (For reference, B0 = {B0:.4e} T)")

## Plot Converted PSDs

In [ ]:
E_COMPS = ["Ex", "Ey", "Ez"]
B_COMPS = ["Bx", "By", "Bz"]
S_COMPS = ["Sx", "Sy", "Sz"]
ENERGY_COMPS = ["uE", "uB"]
COMP_COLORS = {
    "Ex": "C0", "Ey": "C1", "Ez": "C2",
    "Bx": "C3", "By": "C4", "Bz": "C5",
    "Sx": "C0", "Sy": "C1", "Sz": "C2",
    "uE": "C6", "uB": "C7",
}

N_total = len(ds_si.trajectory)
x0s_code = ds_si.x0_code.values
y0s_code = ds_si.y0_code.values

# --- Compute global axis limits ---
all_freqs_min, all_freqs_max = np.inf, -np.inf
e_psd_min, e_psd_max = np.inf, -np.inf
b_psd_min, b_psd_max = np.inf, -np.inf
d_psd_min, d_psd_max = np.inf, -np.inf

for traj_idx in range(N_total):
    n_f = int(ds_si.n_freq.values[traj_idx])
    f = freq_hz[1:n_f]
    if len(f) == 0:
        continue
    all_freqs_min = min(all_freqs_min, f[0])
    all_freqs_max = max(all_freqs_max, f[-1])
    for comp in ALL_COMPS:
        p = ds_si[f"psd_{comp}"].values[traj_idx, 1:n_f]
        p_valid = p[(~np.isnan(p)) & (p > 0)]
        if len(p_valid) == 0:
            continue
        if comp in E_COMPS:
            e_psd_min = min(e_psd_min, p_valid.min())
            e_psd_max = max(e_psd_max, p_valid.max())
        elif comp in B_COMPS:
            b_psd_min = min(b_psd_min, p_valid.min())
            b_psd_max = max(b_psd_max, p_valid.max())
        else:
            d_psd_min = min(d_psd_min, p_valid.min())
            d_psd_max = max(d_psd_max, p_valid.max())

FREQ_LIM  = (all_freqs_min * 0.8, all_freqs_max * 1.2)
E_PSD_LIM = (e_psd_min * 0.3, e_psd_max * 3.0)
B_PSD_LIM = (b_psd_min * 0.3, b_psd_max * 3.0)
D_PSD_LIM = (d_psd_min * 0.3, d_psd_max * 3.0)

print(f"Freq limits [Hz]:      {FREQ_LIM[0]:.2e} -- {FREQ_LIM[1]:.2e}")
print(f"E PSD limits:          {E_PSD_LIM[0]:.2e} -- {E_PSD_LIM[1]:.2e}")
print(f"B PSD limits:          {B_PSD_LIM[0]:.2e} -- {B_PSD_LIM[1]:.2e}")
print(f"Derived PSD limits:    {D_PSD_LIM[0]:.2e} -- {D_PSD_LIM[1]:.2e}")

In [ ]:
for traj_idx in range(N_total):
    n_f = int(ds_si.n_freq.values[traj_idx])
    x0 = x0s_code[traj_idx]
    y0 = y0s_code[traj_idx]
    end = int(ds_si.end_time_idx.values[traj_idx])

    f = freq_hz[:n_f]

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Left: E-field
    ax = axes[0]
    for comp in E_COMPS:
        p = ds_si[f"psd_{comp}"].values[traj_idx, :n_f]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2, label=comp)
    ax.set_xlim(FREQ_LIM)
    ax.set_ylim(E_PSD_LIM)
    ax.set_xlabel("frequency [Hz]")
    ax.set_ylabel(CONV["Ex"][1])
    ax.set_title("E-field")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    # Middle: B-field
    ax = axes[1]
    for comp in B_COMPS:
        p = ds_si[f"psd_{comp}"].values[traj_idx, :n_f]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2, label=comp)
    ax.set_xlim(FREQ_LIM)
    ax.set_ylim(B_PSD_LIM)
    ax.set_xlabel("frequency [Hz]")
    ax.set_ylabel(CONV["Bx"][1])
    ax.set_title("B-field")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    # Right: Poynting + energy
    ax = axes[2]
    for comp in S_COMPS:
        p = ds_si[f"psd_{comp}"].values[traj_idx, :n_f]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2, label=comp)
    for comp in ENERGY_COMPS:
        p = ds_si[f"psd_{comp}"].values[traj_idx, :n_f]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2,
                  linestyle="--", label=comp)
    ax.set_xlim(FREQ_LIM)
    ax.set_ylim(D_PSD_LIM)
    ax.set_xlabel("frequency [Hz]")
    ax.set_ylabel(CONV["Sx"][1])
    ax.set_title("Poynting & Energy")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    fig.suptitle(
        f"Trajectory {traj_idx:02d}  "
        f"(x0={x0:.1f} $d_i$, y0={y0:.1f} $d_i$)  "
        f"{end} steps  [SI units]",
        fontsize=12,
    )
    plt.tight_layout()

    fname = os.path.join(
        PSD_DIR,
        f"psd_si_{traj_idx:02d}_x0_{x0:.1f}_y0_{y0:.1f}.png",
    )
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved {N_total} SI PSD plots to {PSD_DIR}/")

## Save Converted Dataset

In [ ]:
out_path = Path(".") / SAVE_NAME
ds_si.to_netcdf(out_path)
print(f"Saved to {out_path}")
print(ds_si)